# Advanced Problems: Parameterized Decorators and Decorator Factories

These problems build on decorator factories such as `@timed(5)`, nested decorators, closure variables, and metadata preservation with `functools.wraps`. Each problem includes a full solution and runnable checks.

**Best-practice focus:** preserve metadata, validate decorator arguments early, avoid hidden global state, expose useful introspection hooks, support arbitrary `*args` and `**kwargs`, and keep decoration-time behavior separate from call-time behavior.

In [1]:
from functools import wraps
from time import perf_counter, sleep
from collections import defaultdict
import inspect
import statistics
import random

## Problem 1 — Robust parameterized timer

Write a decorator factory `timed(reps=1, *, precision=6, label=None, return_stats=False)` that:

1. Can be used as `@timed(5)` or `@timed(reps=5, precision=4)`.
2. Preserves the wrapped function metadata.
3. Measures a function multiple times and prints the average run time.
4. Returns the original function result by default.
5. If `return_stats=True`, returns `(result, stats)` where `stats` contains `reps`, `avg`, `min`, and `max`.
6. Validates that `reps` is a positive integer and `precision` is a non-negative integer.

In [2]:
def timed(reps=1, *, precision=6, label=None, return_stats=False):
    if not isinstance(reps, int) or reps <= 0:
        raise ValueError("reps must be a positive integer")
    if not isinstance(precision, int) or precision < 0:
        raise ValueError("precision must be a non-negative integer")

    def decorator(fn):
        display_name = label or fn.__qualname__

        @wraps(fn)
        def inner(*args, **kwargs):
            timings = []
            result = None
            for _ in range(reps):
                start = perf_counter()
                result = fn(*args, **kwargs)
                elapsed = perf_counter() - start
                timings.append(elapsed)

            stats = {
                "reps": reps,
                "avg": sum(timings) / reps,
                "min": min(timings),
                "max": max(timings),
            }
            print(f"{display_name}: avg={stats['avg']:.{precision}f}s over {reps} reps")
            return (result, stats) if return_stats else result

        return inner

    return decorator


@timed(3, precision=5, return_stats=True)
def slow_add(a, b):
    sleep(0.01)
    return a + b

value, stats = slow_add(10, 20)
assert value == 30
assert stats["reps"] == 3
assert slow_add.__name__ == "slow_add"
print(stats)

slow_add: avg=0.01038s over 3 reps
{'reps': 3, 'avg': 0.010381100699305534, 'min': 0.010255800560116768, 'max': 0.01056550070643425}


### Solution notes

The outer function validates and stores decorator parameters. The middle function receives the function being decorated. The innermost function receives the runtime arguments of the decorated function. `@wraps(fn)` keeps `__name__`, `__doc__`, `__qualname__`, annotations, and `__wrapped__` useful for introspection.

## Problem 2 — Dual-mode decorator: `@timed` and `@timed(...)`

Create `flex_timed` that supports both forms:

```python
@flex_timed
def f(...): ...

@flex_timed(reps=5)
def g(...): ...
```

It should preserve metadata, print timing information, and reject invalid `reps`.

In [3]:
def flex_timed(_fn=None, *, reps=1, precision=6):
    if not isinstance(reps, int) or reps <= 0:
        raise ValueError("reps must be a positive integer")

    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            timings = []
            result = None
            for _ in range(reps):
                start = perf_counter()
                result = fn(*args, **kwargs)
                timings.append(perf_counter() - start)
            avg = sum(timings) / reps
            print(f"{fn.__qualname__}: avg={avg:.{precision}f}s over {reps} reps")
            return result
        return inner

    if _fn is None:
        return decorator

    if not callable(_fn):
        raise TypeError("@flex_timed must decorate a callable")

    return decorator(_fn)


@flex_timed
def square(x):
    return x * x

@flex_timed(reps=2, precision=5)
def cube(x):
    return x ** 3

assert square(7) == 49
assert cube(3) == 27
assert square.__name__ == "square"
assert cube.__name__ == "cube"

square: avg=0.000002s over 1 reps
cube: avg=0.00000s over 2 reps


### Solution notes

The sentinel argument `_fn=None` distinguishes direct decorator use from factory use. With `@flex_timed`, Python calls `flex_timed(function)`. With `@flex_timed(reps=5)`, Python first calls `flex_timed(reps=5)`, which returns the actual decorator.

## Problem 3 — Call counter with closure state and reset hook

Implement `call_counter(label=None)` that counts how many times each decorated function is called. Requirements:

1. Use closure state, not a global variable.
2. Preserve metadata.
3. Attach `call_count()` and `reset_count()` methods to the wrapper.
4. Work with any positional and keyword arguments.

In [4]:
def call_counter(label=None):
    def decorator(fn):
        count = 0
        display_name = label or fn.__qualname__

        @wraps(fn)
        def inner(*args, **kwargs):
            nonlocal count
            count += 1
            print(f"{display_name} call #{count}")
            return fn(*args, **kwargs)

        def call_count():
            return count

        def reset_count():
            nonlocal count
            count = 0

        inner.call_count = call_count
        inner.reset_count = reset_count
        return inner

    return decorator


@call_counter()
def join_words(*words, sep=" "):
    return sep.join(words)

assert join_words("decorators", "rock") == "decorators rock"
assert join_words("a", "b", "c", sep="-") == "a-b-c"
assert join_words.call_count() == 2
join_words.reset_count()
assert join_words.call_count() == 0

join_words call #1
join_words call #2


### Solution notes

`nonlocal count` makes the wrapper update the variable stored in the enclosing decorator scope. Attaching methods to the wrapper gives callers controlled access to the closure state without exposing a mutable global.

## Problem 4 — Parameterized retry decorator with exception filtering

Write `retry(max_attempts=3, *, exceptions=(Exception,), delay=0, reraise=True)` that retries a failing function. Requirements:

1. Validate `max_attempts >= 1`.
2. Retry only exceptions listed in `exceptions`.
3. Sleep `delay` seconds between failed attempts.
4. If all attempts fail and `reraise=True`, raise the final exception.
5. If all attempts fail and `reraise=False`, return `None`.
6. Preserve metadata.

In [5]:
def retry(max_attempts=3, *, exceptions=(Exception,), delay=0, reraise=True):
    if not isinstance(max_attempts, int) or max_attempts < 1:
        raise ValueError("max_attempts must be an integer >= 1")
    if not isinstance(delay, (int, float)) or delay < 0:
        raise ValueError("delay must be a non-negative number")
    if not isinstance(exceptions, tuple) or not all(isinstance(e, type) and issubclass(e, BaseException) for e in exceptions):
        raise TypeError("exceptions must be a tuple of exception classes")

    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            last_error = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as exc:
                    last_error = exc
                    print(f"attempt {attempt}/{max_attempts} failed: {exc}")
                    if attempt < max_attempts and delay:
                        sleep(delay)

            if reraise:
                raise last_error
            return None

        return inner

    return decorator


state = {"calls": 0}

@retry(max_attempts=4, exceptions=(ValueError,), delay=0)
def flaky():
    state["calls"] += 1
    if state["calls"] < 3:
        raise ValueError("not yet")
    return "ok"

assert flaky() == "ok"
assert state["calls"] == 3
assert flaky.__name__ == "flaky"

attempt 1/4 failed: not yet
attempt 2/4 failed: not yet


### Solution notes

Validation belongs in the outer factory, so configuration mistakes fail at decoration time. Runtime failures belong in the inner wrapper, because they depend on the decorated function call.

## Problem 5 — Memoization decorator with a bounded cache and statistics

Create `memoize(maxsize=None)` with these features:

1. Cache results by function arguments.
2. Support keyword arguments.
3. Preserve metadata.
4. Add `cache_info()` and `cache_clear()` methods.
5. If `maxsize` is set, evict the oldest cached entry when the cache grows beyond `maxsize`.
6. Raise a helpful error when arguments are unhashable.

In [6]:
def memoize(maxsize=None):
    if maxsize is not None and (not isinstance(maxsize, int) or maxsize <= 0):
        raise ValueError("maxsize must be None or a positive integer")

    def make_key(args, kwargs):
        try:
            return args, tuple(sorted(kwargs.items()))
        except TypeError as exc:
            raise TypeError("keyword arguments must be sortable to build a cache key") from exc

    def decorator(fn):
        cache = {}
        order = []
        hits = 0
        misses = 0

        @wraps(fn)
        def inner(*args, **kwargs):
            nonlocal hits, misses
            key = make_key(args, kwargs)
            try:
                hash(key)
            except TypeError as exc:
                raise TypeError("memoized function arguments must be hashable") from exc

            if key in cache:
                hits += 1
                return cache[key]

            misses += 1
            result = fn(*args, **kwargs)
            cache[key] = result
            order.append(key)

            if maxsize is not None and len(cache) > maxsize:
                oldest = order.pop(0)
                cache.pop(oldest, None)

            return result

        def cache_info():
            return {"hits": hits, "misses": misses, "maxsize": maxsize, "currsize": len(cache)}

        def cache_clear():
            nonlocal hits, misses
            cache.clear()
            order.clear()
            hits = misses = 0

        inner.cache_info = cache_info
        inner.cache_clear = cache_clear
        return inner

    return decorator


@memoize(maxsize=3)
def fib(n):
    if n < 3:
        return 1
    return fib(n - 1) + fib(n - 2)

assert fib(10) == 55
info = fib.cache_info()
print(info)
assert info["currsize"] <= 3
fib.cache_clear()
assert fib.cache_info() == {"hits": 0, "misses": 0, "maxsize": 3, "currsize": 0}

{'hits': 7, 'misses': 10, 'maxsize': 3, 'currsize': 3}


### Solution notes

This deliberately implements a small cache manually to exercise closure state and wrapper methods. In production, prefer `functools.lru_cache` unless you need custom behavior.

## Problem 6 — Decorator stack order and metadata inspection

Given two decorator factories below, predict the printed output and the final result before running the cell. Then explain why the order happens.

```python
@tag("outer")
@tag("inner")
def greet(name):
    return f"hello {name}"
```

In [7]:
def tag(name):
    print(f"creating decorator: {name}")

    def decorator(fn):
        print(f"decorating {fn.__name__} with {name}")

        @wraps(fn)
        def inner(*args, **kwargs):
            print(f"enter {name}")
            result = fn(*args, **kwargs)
            print(f"exit {name}")
            return result

        return inner

    return decorator


@tag("outer")
@tag("inner")
def greet(name):
    """Return a greeting."""
    return f"hello {name}"

result = greet("Ada")
print("result:", result)
print("name:", greet.__name__)
print("doc:", greet.__doc__)
print("wrapped chain exists:", hasattr(greet, "__wrapped__"))

creating decorator: outer
creating decorator: inner
decorating greet with inner
decorating greet with outer
enter outer
enter inner
exit inner
exit outer
result: hello Ada
name: greet
doc: Return a greeting.
wrapped chain exists: True


### Solution

Decorator factory expressions are evaluated from top to bottom, so `tag("outer")` is created before `tag("inner")`. Actual decoration is applied from bottom to top, equivalent to:

```python
greet = tag("outer")(tag("inner")(greet))
```

At call time, the outer wrapper runs first, then the inner wrapper, then the original function. Because both wrappers use `@wraps`, the visible function name and docstring still look like the original `greet`.

## Problem 7 — Type-enforcing decorator factory using annotations

Create `enforce_types(strict_return=True)` that checks runtime arguments and the return value against simple annotations. Requirements:

1. Use `inspect.signature` to bind arguments.
2. Ignore parameters without annotations.
3. Check return type only when `strict_return=True` and a return annotation exists.
4. Preserve metadata.
5. Raise `TypeError` with a helpful message.

In [8]:
def enforce_types(*, strict_return=True):
    def decorator(fn):
        sig = inspect.signature(fn)
        annotations = fn.__annotations__

        @wraps(fn)
        def inner(*args, **kwargs):
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()

            for name, value in bound.arguments.items():
                expected = annotations.get(name, inspect.Signature.empty)
                if expected is inspect.Signature.empty:
                    continue
                if not isinstance(value, expected):
                    raise TypeError(f"{name} must be {expected.__name__}, got {type(value).__name__}")

            result = fn(*args, **kwargs)
            expected_return = annotations.get("return", inspect.Signature.empty)
            if strict_return and expected_return is not inspect.Signature.empty:
                if not isinstance(result, expected_return):
                    raise TypeError(f"return must be {expected_return.__name__}, got {type(result).__name__}")

            return result

        return inner

    return decorator


@enforce_types(strict_return=True)
def repeat(text: str, times: int = 2) -> str:
    return text * times

assert repeat("ha", 3) == "hahaha"
try:
    repeat("ha", "3")
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("TypeError was expected")

times must be int, got str


### Solution notes

`inspect.signature(...).bind(...)` maps runtime positional and keyword arguments back to parameter names. This is safer than manually zipping `args` with parameter names.

## Problem 8 — Compose a production-style decorator pipeline

Build a function decorated with `retry`, `call_counter`, and `timed`. Then reason about stack order. Use this order:

```python
@timed(2)
@call_counter("unstable_api")
@retry(max_attempts=3, exceptions=(RuntimeError,), reraise=False)
def unstable_api(seed):
    ...
```

Requirements:

1. The function sometimes fails with `RuntimeError`.
2. The retry logic should happen inside the call counter and timer.
3. The timer should measure the whole operation, including retries.
4. Show that metadata is still preserved.

In [9]:
@timed(2, precision=5)
@call_counter("unstable_api")
@retry(max_attempts=3, exceptions=(RuntimeError,), reraise=False)
def unstable_api(seed):
    random.seed(seed + unstable_api.call_count())
    if random.random() < 0.4:
        raise RuntimeError("temporary outage")
    return {"seed": seed, "status": "ok"}

response = unstable_api(10)
print("response:", response)
print("calls:", unstable_api.call_count())
print("name:", unstable_api.__name__)
assert unstable_api.__name__ == "unstable_api"

unstable_api call #1
unstable_api call #2
unstable_api: avg=0.00018s over 2 reps
response: {'seed': 10, 'status': 'ok'}
calls: 2
name: unstable_api


### Solution notes

Decoration is bottom-up, so the original function is first wrapped by `retry`, then by `call_counter`, then by `timed`. Calling the final function enters `timed` first, so the timer measures everything below it, including counting and retries. Because every layer uses `@wraps`, metadata and wrapper-added attributes remain easier to inspect.

## Challenge extensions

1. Modify `timed` to accept a custom logger function instead of always printing.
2. Modify `memoize` to implement true LRU behavior instead of oldest-insertion eviction.
3. Extend `enforce_types` to support `list[int]`, `dict[str, int]`, and `typing.Union`.
4. Create a class-based version of `call_counter` and compare it with the closure-based version.
5. Write tests using `pytest` for every decorator factory above.